# H&M Data Cleaning — 4 Versions as Parquet

Creates 4 cleaned versions of the dataset saved as **parquet** files (smaller, faster, keeps data types):
- **normal_clean/** — NaNs filled with fixed values (0, UNKNOWN, median age, etc.)
- **mean_version/** — numeric NaNs filled with MEAN, categorical NaNs filled with MODE
- **mode_version/** — ALL NaNs filled with MODE
- **median_version/** — numeric NaNs filled with MEDIAN, categorical NaNs filled with MODE

Only columns that actually HAVE NaNs get filled. Columns with no NaNs are left alone.
Numeric columns get mean or median. Categorical/text columns always get mode.

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_DIR = r'A:\IDEs\PyCharm\hAnd-m-PersonalizedFashionRecommendations\dataset'
OUTPUT_DIR = r'A:\IDEs\PyCharm\hAnd-m-PersonalizedFashionRecommendations\clean dataset'

for sub in ['normal_clean', 'mean_version', 'mode_version', 'median_version']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)

print('Output folders created:')
for sub in ['normal_clean', 'mean_version', 'mode_version', 'median_version']:
    print(f'  {os.path.join(OUTPUT_DIR, sub)}')

In [ ]:
print('Loading articles...')
articles = pd.read_csv(os.path.join(DATA_DIR, 'articles.csv'))
articles['article_id'] = articles['article_id'].astype(str)
print(f'  articles: {articles.shape}')

print('Loading customers...')
customers = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
customers['customer_id'] = customers['customer_id'].astype(str)
print(f'  customers: {customers.shape}')

print('Loading transactions (3.2GB, takes a minute)...')
transactions = pd.read_csv(os.path.join(DATA_DIR, 'transactions_train.csv'))
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
transactions['article_id'] = transactions['article_id'].astype(str)
transactions['customer_id'] = transactions['customer_id'].astype(str)
transactions = transactions.dropna(subset=['t_dat', 'customer_id', 'article_id', 'price'])
print(f'  transactions: {transactions.shape}')

print('\nAll files loaded and ready.')

In [ ]:
def fill_nans(df, strategy):
    """Fill NaN values. numeric gets mean/median, categorical gets mode. mode gets all mode."""
    df = df.copy()
    null_cols = df.columns[df.isnull().any()].tolist()
    for col in null_cols:
        is_numeric = pd.api.types.is_numeric_dtype(df[col])
        if strategy == 'mode':
            fill_val = df[col].mode()[0] if not df[col].mode().empty else 0
            df[col] = df[col].fillna(fill_val)
            print(f'    {col}: filled with mode = {fill_val}')
        elif strategy == 'mean':
            if is_numeric:
                fill_val = df[col].mean()
                df[col] = df[col].fillna(fill_val)
                print(f'    {col}: filled with mean = {fill_val:.2f}')
            else:
                fill_val = df[col].mode()[0] if not df[col].mode().empty else ''
                df[col] = df[col].fillna(fill_val)
                print(f'    {col}: filled with mode = {fill_val} (categorical)')
        elif strategy == 'median':
            if is_numeric:
                fill_val = df[col].median()
                df[col] = df[col].fillna(fill_val)
                print(f'    {col}: filled with median = {fill_val:.2f}')
            else:
                fill_val = df[col].mode()[0] if not df[col].mode().empty else ''
                df[col] = df[col].fillna(fill_val)
                print(f'    {col}: filled with mode = {fill_val} (categorical)')
    return df

def make_normal_clean(articles, customers, transactions):
    """Fixed fills: FN/Active to 0, status to UNKNOWN, news to NONE, age to median, desc to empty, postal to UNKNOWN"""
    a = articles.copy()
    c = customers.copy()
    t = transactions.copy()
    a['detail_desc'] = a['detail_desc'].fillna('')
    c['FN'] = c['FN'].fillna(0).astype(int)
    c['Active'] = c['Active'].fillna(0).astype(int)
    c['club_member_status'] = c['club_member_status'].fillna('UNKNOWN')
    c['fashion_news_frequency'] = c['fashion_news_frequency'].fillna('NONE')
    c['age'] = c['age'].fillna(c['age'].median())
    c['postal_code'] = c['postal_code'].fillna('UNKNOWN')
    return a, c, t

print('Functions ready.')

In [ ]:
for name, df in [('articles', articles), ('customers', customers), ('transactions', transactions)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f'\n{name} NaN columns:')
    if len(nulls) > 0:
        for col, count in nulls.items():
            print(f'  {col}: {count} NaNs (dtype: {df[col].dtype})')
    else:
        print('  No NaNs')

## Version 1: normal_clean (fixed fills)

In [ ]:
print('=== NORMAL CLEAN VERSION ===')
a_normal, c_normal, t_normal = make_normal_clean(articles, customers, transactions)
print(f'articles NaNs: {a_normal.isnull().sum().sum()}')
print(f'customers NaNs: {c_normal.isnull().sum().sum()}')
print(f'transactions NaNs: {t_normal.isnull().sum().sum()}')

In [ ]:
normal_dir = os.path.join(OUTPUT_DIR, 'normal_clean')
a_normal.to_parquet(os.path.join(normal_dir, 'articles.parquet'), index=False)
c_normal.to_parquet(os.path.join(normal_dir, 'customers.parquet'), index=False)
t_normal.to_parquet(os.path.join(normal_dir, 'transactions.parquet'), index=False)
print(f'Saved normal_clean (parquet) to: {normal_dir}')
print(f'  articles.parquet: {a_normal.shape}')
print(f'  customers.parquet: {c_normal.shape}')
print(f'  transactions.parquet: {t_normal.shape}')

## Version 2: Mean (numeric → mean, categorical → mode)

In [ ]:
print('=== MEAN VERSION ===')
print('--- articles ---')
a_mean = fill_nans(articles, 'mean')
print('--- customers ---')
c_mean = fill_nans(customers, 'mean')
t_mean = transactions.copy()
print(f'\nRemaining NaNs: articles={a_mean.isnull().sum().sum()}, customers={c_mean.isnull().sum().sum()}')

In [ ]:
mean_dir = os.path.join(OUTPUT_DIR, 'mean_version')
a_mean.to_parquet(os.path.join(mean_dir, 'articles.parquet'), index=False)
c_mean.to_parquet(os.path.join(mean_dir, 'customers.parquet'), index=False)
t_mean.to_parquet(os.path.join(mean_dir, 'transactions.parquet'), index=False)
print(f'Saved mean_version (parquet) to: {mean_dir}')
print(f'  articles.parquet: {a_mean.shape}')
print(f'  customers.parquet: {c_mean.shape}')
print(f'  transactions.parquet: {t_mean.shape}')

## Version 3: Mode (all columns → mode)

In [ ]:
print('=== MODE VERSION ===')
print('--- articles ---')
a_mode = fill_nans(articles, 'mode')
print('--- customers ---')
c_mode = fill_nans(customers, 'mode')
t_mode = transactions.copy()
print(f'\nRemaining NaNs: articles={a_mode.isnull().sum().sum()}, customers={c_mode.isnull().sum().sum()}')

In [ ]:
mode_dir = os.path.join(OUTPUT_DIR, 'mode_version')
a_mode.to_parquet(os.path.join(mode_dir, 'articles.parquet'), index=False)
c_mode.to_parquet(os.path.join(mode_dir, 'customers.parquet'), index=False)
t_mode.to_parquet(os.path.join(mode_dir, 'transactions.parquet'), index=False)
print(f'Saved mode_version (parquet) to: {mode_dir}')
print(f'  articles.parquet: {a_mode.shape}')
print(f'  customers.parquet: {c_mode.shape}')
print(f'  transactions.parquet: {t_mode.shape}')

## Version 4: Median (numeric → median, categorical → mode)

In [ ]:
print('=== MEDIAN VERSION ===')
print('--- articles ---')
a_median = fill_nans(articles, 'median')
print('--- customers ---')
c_median = fill_nans(customers, 'median')
t_median = transactions.copy()
print(f'\nRemaining NaNs: articles={a_median.isnull().sum().sum()}, customers={c_median.isnull().sum().sum()}')

In [ ]:
median_dir = os.path.join(OUTPUT_DIR, 'median_version')
a_median.to_parquet(os.path.join(median_dir, 'articles.parquet'), index=False)
c_median.to_parquet(os.path.join(median_dir, 'customers.parquet'), index=False)
t_median.to_parquet(os.path.join(median_dir, 'transactions.parquet'), index=False)
print(f'Saved median_version (parquet) to: {median_dir}')
print(f'  articles.parquet: {a_median.shape}')
print(f'  customers.parquet: {c_median.shape}')
print(f'  transactions.parquet: {t_median.shape}')

## Verify — all 4 versions have zero NaNs

In [ ]:
versions = {
    'normal_clean': (a_normal, c_normal, t_normal),
    'mean_version': (a_mean, c_mean, t_mean),
    'mode_version': (a_mode, c_mode, t_mode),
    'median_version': (a_median, c_median, t_median),
}

print(f'{"Version":<15} {"articles":<10} {"customers":<10} {"transactions":<12}')
print('-' * 50)
for vname, (a, c, t) in versions.items():
    print(f'{vname:<15} {a.isnull().sum().sum():<10} {c.isnull().sum().sum():<10} {t.isnull().sum().sum():<12}')

print('\nAll should show 0.')

In [ ]:
# Show file size comparison: CSV vs Parquet
print('File size comparison (CSV vs Parquet):')
print(f'{"File":<25} {"CSV (MB)":<12} {"Parquet (MB)":<15} {"Reduction"}')
print('-' * 65)

csv_sizes = {
    'articles': os.path.getsize(os.path.join(DATA_DIR, 'articles.csv')) / 1e6,
    'customers': os.path.getsize(os.path.join(DATA_DIR, 'customers.csv')) / 1e6,
    'transactions': os.path.getsize(os.path.join(DATA_DIR, 'transactions_train.csv')) / 1e6,
}
pq_sizes = {
    'articles': os.path.getsize(os.path.join(normal_dir, 'articles.parquet')) / 1e6,
    'customers': os.path.getsize(os.path.join(normal_dir, 'customers.parquet')) / 1e6,
    'transactions': os.path.getsize(os.path.join(normal_dir, 'transactions.parquet')) / 1e6,
}
for name in ['articles', 'customers', 'transactions']:
    csv_mb = csv_sizes[name]
    pq_mb = pq_sizes[name]
    reduction = (1 - pq_mb / csv_mb) * 100
    print(f'{name:<25} {csv_mb:<12.1f} {pq_mb:<15.1f} {reduction:.0f}%')

## Summary

4 versions saved as **parquet** to `clean dataset/`:

| Version | Numeric NaNs | Categorical NaNs |
|---------|-------------|------------------|
| normal_clean | Fixed values (0, median age) | Fixed values (UNKNOWN, NONE) |
| mean_version | Filled with MEAN | Filled with MODE |
| mode_version | Filled with MODE | Filled with MODE |
| median_version | Filled with MEDIAN | Filled with MODE |

Each version has: articles.parquet, customers.parquet, transactions.parquet

Parquet keeps data types (datetime stays datetime, strings stay strings) and is much smaller than CSV.